In [18]:
import pandas as pd
from pathlib import Path

# Load experiment 1 datasets
exp_dir = Path('../data/gold/experiment_1')
processes = [p for p in exp_dir.iterdir() if p.is_dir()]

results = []
for p in processes:

    print(p)
    dataset_dir = p / 'datasets'
    if not dataset_dir.exists():
        continue
    
    df_exp_path = dataset_dir / 'df_expanded.parquet'
    if not df_exp_path.exists():
        continue
        
    df = pd.read_parquet(df_exp_path)
    
    # Energy variables (contain 'energy' but NOT 'ef')
    energy_vars = [col for col in df.columns if 'to_model' in col.lower()]

    print(energy_vars)

    num_cols_energy = len(energy_vars) - 1

    
    
    # Number of cases
    case_col = 'case_id' if 'case_id' in df.columns else df.columns[0]
    num_cases = df[case_col].nunique() if case_col in df.columns else None
    
    # Number of activities
    activity_col = 'activity_log'
    num_activities = df[activity_col].nunique() if activity_col in df.columns else None

    # Max and min date, complete duration
    time_cols = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]
    time_col = time_cols[0] if time_cols else None
    if 'timestamp' in df.columns:
        time_col = 'timestamp'
    elif 'time:timestamp' in df.columns:
        time_col = 'time:timestamp'
        
    if time_col:
        # Try converting to datetime if not already
        df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
        min_date = df[time_col].min()
        max_date = df[time_col].max()
        complete_date = max_date - min_date
        # Convert timedelta to total hours (rounded to 2 decimal places for readability)
        hours = round(complete_date.total_seconds() / 3600, 2) if pd.notnull(complete_date) else None
    else:
        min_date, max_date, hours = None, None, None
        
    # Number of rows in expanded
    num_rows = len(df)
    
    results.append({
        'dataset': p.name,
        'number of cases': num_cases,
        'number of activities': num_activities,
        'number of time series': num_cols_energy,
        'hours': int(round(hours)) if hours is not None else None,
        #'observations': num_rows
    })

# Sort and display the DataFrame
df_info = pd.DataFrame(results).sort_values(by='dataset')
display(df_info)

# Output a single LaTeX table for all datasets
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Dataset Information Overview}")
print(df_info.to_latex(index=False).replace('_', '\\_'))
print("\\end{table}")


../data/gold/experiment_1/process_4
['temp_Auslauf_EG_(WT2)_5s_energy_to_model', 'temp_Einlauf_EG_(WT_2)_5s_energy_to_model', 'flow_Kuehlturmwasser_30120FT701_5s_energy_to_model', 'flow_Kaltwasser_(WT7)_5s_energy_to_model', 'Vorlaufpumpe_30110FT301_5s_energy_to_model', 'vor_Vorwärmer_(WT_2)_5s_energy_to_model', 'Kuehlturmwassertemp_(WT6)_5s_energy_to_model', 'Kaltwassertemp_(WT_7)_5s_energy_to_model', 'nach_Kuehler_(WT7)_5s_energy_to_model', 'temp_nach_Kuehlturmkuehler_(WT6)_5s_energy_to_model', 'Fuellstand_Steriltank_30140LT001_5s_energy_to_model', 'Fuellstand_Steriltank_30141LT001_5s_energy_to_model', 'flow_Dampf_WT3a/5a)_5s_energy_to_model', 'flow_Heisswasser_30120FT721(WT5a)_5s_energy_to_model', 'temp_nach_WR2,_vor_Druckerhoehungspumpe_(WT4)_5s_energy_to_model', 'temp_nach_Erhitzer_(WT5)_5s_energy_to_model', 'temp_nach_WR2_(WT2)_5s_energy_to_model', 'temp_nach_Austauscher_2_(WT4)_5s_energy_to_model', 'Druck_HW_Anwaermer_(WT3a)_5s_energy_to_model', 'temp_HW_Anwaermer_(WT3a)_5s_energ

,dataset,number of cases,number of activities,number of time series,hours
1,process_1,34585,20,5,155
2,process_2,368493,12,4,512
3,process_3,316,5,22,10043
0,process_4,53,19,27,705


\begin{table}[h]
\centering
\caption{Dataset Information Overview}
\begin{tabular}{lrrrr}
\toprule
dataset & number of cases & number of activities & number of time series & hours \\
\midrule
process\_1 & 34585 & 20 & 5 & 155 \\
process\_2 & 368493 & 12 & 4 & 512 \\
process\_3 & 316 & 5 & 22 & 10043 \\
process\_4 & 53 & 19 & 27 & 705 \\
\bottomrule
\end{tabular}

\end{table}


In [19]:
# ── Config ────────────────────────────────────────────────────────────────────
# Set the experiment number to load (picks the latest run automatically)
EXPERIMENT = 405
# Which evaluation split to display: 'train' or 'test'
# Both live in the same row — train_* vs test_* column prefixes.
SPLIT = 'test'

In [20]:
import pandas as pd
from pathlib import Path

results_root = Path('..') / 'results'

# Find the latest run folder for the given experiment
prefix = f'experiment_{EXPERIMENT}_'
runs = sorted([d for d in results_root.iterdir() if d.is_dir() and d.name.startswith(prefix)])
assert runs, f'No runs found for experiment {EXPERIMENT}'
run_dir = runs[-1]
print(f'Loading: {run_dir.name}')

df = pd.read_parquet(run_dir / 'process_eval_results.parquet')
# The 'split' column marks the training set used (always 'TRAIN' or 'ALL DATA').
# Train vs test evaluation results are distinguished by column prefix: train_* vs test_*.
# Check the requested prefix exists.
split_prefix = SPLIT.lower() + '_'
available_prefixes = set()
for c in df.columns:
    for p in ('train_', 'test_'):
        if c.startswith(p):
            available_prefixes.add(p.rstrip('_'))
print(f'Available column prefixes: {sorted(available_prefixes)}')
assert split_prefix.rstrip('_') in available_prefixes, \
    f'No columns with prefix "{split_prefix}" found. Available: {sorted(available_prefixes)}'

print(f'Split prefix: {split_prefix} | {len(df)} rows | processes: {sorted(df["process"].unique())}')

Loading: experiment_405_20260625_111926
Available column prefixes: ['test', 'train']
Split prefix: test_ | 40 rows | processes: ['process_1', 'process_2', 'process_3', 'process_4']


In [21]:
# ── Mode display labels (mirrors modelling.py _display_mode) ─────────────────
def display_mode(m):
    m = str(m)
    if not m.startswith('petri_net_'):
        return m
    rest = m[len('petri_net_'):]
    if rest.endswith('_ml_plus_global'):
        return rest[:-len('_ml_plus_global')] + ' / ml_global'
    if rest.endswith('_ml_plus_per_act'):
        return rest[:-len('_ml_plus_per_act')] + ' / ml_local'
    return rest + ' / baseline'

df['mode_label'] = df['mode'].map(display_mode)

In [22]:
# ── Metrics for the results table ────────────────────────────────────────────
# activity_duration_mae:  MAE of per-activity mean durations in minutes
# activity_duration_wape: weighted abs % error of per-activity mean durations (0 = perfect)
# activity_duration_rmse: RMSE of per-activity mean durations in minutes
# edge_f1_score:          control-flow edge F1 on directly-follows graph (1 = perfect)
# fitness / precision:    conformance values via token replay (1 = perfect)

split_prefix = SPLIT.lower() + '_'

# Duration metrics: prefer MAE/WAPE/RMSE (new pipeline), fall back to MAPE for old parquets
_has_wape = (split_prefix + 'duration_metrics_activity_duration_wape') in df.columns

if _has_wape:
    _dur_cols = [
        split_prefix + 'duration_metrics_activity_duration_mae',
        split_prefix + 'duration_metrics_activity_duration_wape',
        split_prefix + 'duration_metrics_activity_duration_rmse',
    ]
    _dur_labels = {
        split_prefix + 'duration_metrics_activity_duration_mae':  r'\makecell{Duration\\MAE (min)}',
        split_prefix + 'duration_metrics_activity_duration_wape': r'\makecell{Duration\\WAPE}',
        split_prefix + 'duration_metrics_activity_duration_rmse': r'\makecell{Duration\\RMSE (min)}',
    }
else:
    _dur_cols = [split_prefix + 'duration_metrics_activity_duration_error']
    _dur_labels = {split_prefix + 'duration_metrics_activity_duration_error': r'\makecell{Duration\\MAPE}'}

METRIC_BASES_ORDERED = (
    _dur_cols
    + [
        split_prefix + 'control_flow_metrics_edge_f1_score',
        split_prefix + 'conformance_metrics_fitness',
        split_prefix + 'conformance_metrics_precision',
    ]
)
metric_cols = [c for c in METRIC_BASES_ORDERED if c in df.columns]

METRIC_LABELS = {
    **_dur_labels,
    split_prefix + 'control_flow_metrics_edge_f1_score': r'\makecell{Edge\\F1 Score}',
    split_prefix + 'conformance_metrics_fitness':        r'\makecell{Fitness}',
    split_prefix + 'conformance_metrics_precision':      r'\makecell{Precision}',
}

processes = sorted(df['process'].unique())
print(f'Duration cols: {_dur_cols}')
print(f'All metrics: {metric_cols}')
print(f'Processes: {processes}')
print(f'Modes: {df["mode_label"].unique().tolist()}')

Duration cols: ['test_duration_metrics_activity_duration_mae', 'test_duration_metrics_activity_duration_wape', 'test_duration_metrics_activity_duration_rmse']
All metrics: ['test_duration_metrics_activity_duration_mae', 'test_duration_metrics_activity_duration_wape', 'test_duration_metrics_activity_duration_rmse', 'test_control_flow_metrics_edge_f1_score', 'test_conformance_metrics_fitness', 'test_conformance_metrics_precision']
Processes: ['process_1', 'process_2', 'process_3', 'process_4']
Modes: ['statistical', 'alpha / baseline', 'heuristic / baseline', 'inductive / baseline', 'alpha / ml_global', 'alpha / ml_local', 'heuristic / ml_global', 'heuristic / ml_local', 'inductive / ml_global', 'inductive / ml_local']


In [23]:
# ── Build table: rows=(process, process model, duration pred), cols=metrics ───

def pick_best_algo(proc_df, split_prefix):
    """Best petri net algo among {heuristic, inductive, ilp} by combined metric score."""
    candidates = ['heuristic', 'inductive', 'ilp']
    base_modes = [f'petri_net_{a}' for a in candidates]
    sub = proc_df[proc_df['mode'].isin(base_modes)]
    if sub.empty:
        return candidates[0]
    _score_cols = [c for c in [
        split_prefix + 'conformance_metrics_fitness',
        split_prefix + 'conformance_metrics_precision',
        split_prefix + 'control_flow_metrics_edge_f1_score',
    ] if c in sub.columns]
    scores = pd.Series(0.0, index=sub.index)
    for c in _score_cols:
        scores = scores + sub[c].fillna(0)
    best_mode = sub.loc[scores.idxmax(), 'mode']
    return best_mode.replace('petri_net_', '')

def make_mode_label(mode, best_algo):
    pn = f'{best_algo} petri net'
    if mode == 'petri_net_alpha':
        return 'alpha petri net / baseline'
    if mode == f'petri_net_{best_algo}':
        return f'{pn} / baseline'
    if mode == f'petri_net_{best_algo}_ml_plus_global':
        return f'{pn} / ml_global'
    if mode == f'petri_net_{best_algo}_ml_plus_per_act':
        return f'{pn} / ml_local'
    return mode

def split_mode(label):
    if ' / ' in label:
        left, right = label.split(' / ', 1)
        return left.strip(), right.strip()
    return label, '—'

frames = []
for proc in processes:
    sub = df[df['process'] == proc].copy()
    best_algo = pick_best_algo(sub, split_prefix)
    print(f'{proc}: best_algo = {best_algo}')

    modes_to_show = [
        'petri_net_alpha',
        f'petri_net_{best_algo}',
        f'petri_net_{best_algo}_ml_plus_global',
        f'petri_net_{best_algo}_ml_plus_per_act',
    ]
    sub = sub[sub['mode'].isin(modes_to_show)].copy()
    sub['mode_label'] = sub['mode'].apply(lambda m: make_mode_label(m, best_algo))

    # ml_local first, ml_global second, baseline (best net) third, alpha last
    pn = f'{best_algo} petri net'
    row_order = [
        f'{pn} / ml_local',
        f'{pn} / ml_global',
        f'{pn} / baseline',
        'alpha petri net / baseline',
    ]
    sub = sub.set_index('mode_label')[metric_cols]
    sub = sub.reindex([r for r in row_order if r in sub.index])
    sub.columns = [METRIC_LABELS.get(c, c) for c in sub.columns]
    pmodel, dpred = zip(*[split_mode(m) for m in sub.index])
    sub.index = pd.MultiIndex.from_arrays(
        [[proc] * len(sub), list(pmodel), list(dpred)],
        names=['Process', 'Process Model', 'Duration Pred.']
    )
    frames.append(sub)

table = pd.concat(frames)
table

process_1: best_algo = heuristic
process_2: best_algo = heuristic
process_3: best_algo = heuristic
process_4: best_algo = heuristic


\makecell{Duration\\MAE (min)}  \
Process   Process Model       Duration Pred.                                   
process_1 heuristic petri net ml_local                              0.166650   
                              ml_global                             0.186690   
                              baseline                              0.959709   
          alpha petri net     baseline                              0.959709   
process_2 heuristic petri net ml_local                              3.462922   
                              ml_global                             3.579715   
                              baseline                              5.834466   
          alpha petri net     baseline                             26.849147   
process_3 heuristic petri net ml_local                             68.562257   
                              ml_global                            70.386396   
                              baseline                             81.027154   
          alpha petri net     baseline                            176.362338   
process_4 heuristic petri net ml_local                              5.411805   
                              ml_global                             4.931913   
                              baseline                              4.889296   
          alpha petri net     baseline                              8.199242   

                                              \makecell{Duration\\WAPE}  \
Process   Process Model       Duration Pred.                              
process_1 heuristic petri net ml_local                         0.063759   
                              ml_global                        0.079681   
                              baseline                         0.354377   
          alpha petri net     baseline                         0.354377   
process_2 heuristic petri net ml_local                         0.245965   
                              ml_global                        0.292246   
                              baseline                         0.416819   
          alpha petri net     baseline                         0.526598   
process_3 heuristic petri net ml_local                         0.569755   
                              ml_global                        0.587406   
                              baseline                         0.670858   
          alpha petri net     baseline                         0.746067   
process_4 heuristic petri net ml_local                         0.525239   
                              ml_global                        0.478663   
                              baseline                         0.474527   
          alpha petri net     baseline                         0.441220   

                                              \makecell{Duration\\RMSE (min)}  \
Process   Process Model       Duration Pred.                                    
process_1 heuristic petri net ml_local                               0.238449   
                              ml_global                              0.289424   
                              baseline                               1.614009   
          alpha petri net     baseline                               1.614009   
process_2 heuristic petri net ml_local                               6.404262   
                              ml_global                              7.872307   
                              baseline                              11.251598   
          alpha petri net     baseline                              32.988552   
process_3 heuristic petri net ml_local                             137.175716   
                              ml_global                            135.657867   
                              baseline                             151.097102   
          alpha petri net     baseline                             243.955387   
process_4 heuristic petri net ml_local                              10.637022   
                              ml_global

In [24]:
# ── LaTeX output ──────────────────────────────────────────────────────────────
# Requires \usepackage{makecell}, \usepackage{booktabs}, \usepackage{multirow} in preamble.
import re

def _is_higher_better(col_label):
    return any(kw in col_label for kw in ['F1 Score', 'Fitness', 'Precision'])

# ── Bold best value per metric per process ────────────────────────────────────
str_table = pd.DataFrame(index=table.index, columns=table.columns, dtype=object)
for proc in processes:
    proc_mask = table.index.get_level_values('Process') == proc
    proc_rows = table[proc_mask]
    for col in table.columns:
        vals = proc_rows[col].dropna()
        if vals.empty:
            for idx in proc_rows.index:
                str_table.loc[idx, col] = ''
            continue
        best_val = vals.max() if _is_higher_better(col) else vals.min()
        for idx in proc_rows.index:
            val = table.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ''
                continue
            fmt = f'{val:.3f}'
            str_table.loc[idx, col] = (r'\textbf{' + fmt + r'}') if abs(val - best_val) < 1e-6 else fmt

# ── Column format: vertical lines between metrics ─────────────────────────────
n_metrics = len(table.columns)
col_format = 'lll|' + '|'.join(['c'] * n_metrics)

# ── Generate LaTeX ─────────────────────────────────────────────────────────────
latex = str_table.to_latex(
    multicolumn=True,
    multicolumn_format='c',
    multirow=True,
    escape=False,
    column_format=col_format,
    caption=(
        f'Results for experiment {EXPERIMENT} ({SPLIT} evaluation). '
        r'Rows: alpha = alpha miner (baseline); best petri net = best of heuristic/inductive per process. '
        r'Activity JS Div.\ and Duration WAPE/MAE/RMSE: lower is better. '
        r'Edge F1 Score, Fitness, Precision: higher is better (1 = best). '
        r'\textbf{Bold} = best value per process per metric.'
    ),
    label=f'tab:exp{EXPERIMENT}_{SPLIT}',
    position='H',
)

# Remove \cline lines (generated by pandas for MultiIndex, replaced by \midrule below)
latex = re.sub(r'\s*\\cline\{[^}]+\}', '', latex)

# Insert \midrule between process groups (detect first row of each group via \multirow + process_)
latex_lines = latex.split('\n')
new_lines = []
proc_count = 0
for line in latex_lines:
    if re.search(r'\\multirow.*\{process_', line):
        if proc_count > 0:
            new_lines.append(r'\midrule')
        proc_count += 1
    new_lines.append(line)
latex = '\n'.join(new_lines)

print(latex.replace('_', r'\_'))

\begin{table}[H]
\caption{Results for experiment 405 (test evaluation). Rows: alpha = alpha miner (baseline); best petri net = best of heuristic/inductive per process. Activity JS Div.\ and Duration WAPE/MAE/RMSE: lower is better. Edge F1 Score, Fitness, Precision: higher is better (1 = best). \textbf{Bold} = best value per process per metric.}
\label{tab:exp405\_test}
\begin{tabular}{lll|c|c|c|c|c|c}
\toprule
 &  &  & \makecell{Duration\\MAE (min)} & \makecell{Duration\\WAPE} & \makecell{Duration\\RMSE (min)} & \makecell{Edge\\F1 Score} & \makecell{Fitness} & \makecell{Precision} \\
Process & Process Model & Duration Pred. &  &  &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{process\_1} & \multirow[t]{3}{*}{heuristic petri net} & ml\_local & \textbf{0.167} & \textbf{0.064} & \textbf{0.238} & \textbf{0.615} & \textbf{1.000} & \textbf{1.000} \\
 &  & ml\_global & 0.187 & 0.080 & 0.289 & \textbf{0.615} & \textbf{1.000} & \textbf{1.000} \\
 &  & baseline & 0.960 & 0.354 & 1.614 & \textbf{0.61

# Evaluation energy

In [25]:
df_energy_results = pd.read_parquet(run_dir / 'summary_by_approach.parquet')

df_energy_results = df_energy_results[['Approach', 'MAE_TEST', 'RMSE_TEST', 'WAPE_TEST']]

df_energy_results

,Approach,MAE_TEST,RMSE_TEST,WAPE_TEST
0,Baseline,95.9369,109.6762,15.4005
1,DTW + Ext. Factors + Prev Act,43.0029,52.5833,5.8430
2,DTW + Ext. Factors + Prev Act (autoreg),63.6681,79.8069,9.5495
3,DTW + Seq2Seq,88.2583,110.0211,14.4082
4,DTW + Seq2Seq + Ext. Factors + Prev Act,71.7425,85.3488,8.3962
5,DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg),106.1971,126.2402,21.2713
6,DTW + pos,81.6115,98.8294,13.1292


In [26]:
import pandas as pd

# Load data
df_energy_results = pd.read_parquet(run_dir / "summary_train_test.parquet")

# Group by Process and Approach, then take median of the selected metrics
df_grouped = (
    df_energy_results
    .groupby(["Process", "Approach"])[["MAE_TEST", "RMSE_TEST", "WAPE_TEST"]]
    .median()
    .reset_index()
)

df_grouped

,Process,Approach,MAE_TEST,RMSE_TEST,WAPE_TEST
0,process_1,Baseline,0.39270,0.39755,55.83590
1,process_2,Baseline,98.48340,113.05650,153.15250
2,process_2,DTW + Ext. Factors + Prev Act,113.08900,122.67610,49.51420
3,process_2,DTW + Ext. Factors + Prev Act (autoreg),113.84950,123.37510,46.21940
4,process_2,DTW + Seq2Seq,79.95360,98.10700,60.94010
5,process_2,DTW + Seq2Seq + Ext. Factors + Prev Act,81.72670,100.37410,76.49770
6,process_2,DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg),81.74820,100.38120,73.86650
7,process_2,DTW + pos,87.84580,97.26630,52.32530
8,process_3,Baseline,411.58990,418.53410,19.00690
9,process_3,DTW + Ext. Factors + Prev Act,142.88180,150.05700,7.20990


In [27]:
import pandas as pd
import re

# Load data
df_energy_results = pd.read_parquet(run_dir / "summary_train_test.parquet")

# Group by Process and Approach, then take median
table = (
    df_energy_results
    .groupby(["Process", "Approach"])[["MAE_TEST", "RMSE_TEST", "WAPE_TEST"]]
    .median()
    .sort_index()
)

processes = table.index.get_level_values("Process").unique()

# Lower is better for all these metrics
def _is_higher_better(col_label):
    return False

# ── Bold best value per metric per process ────────────────────────────────────
str_table = pd.DataFrame(index=table.index, columns=table.columns, dtype=object)

for proc in processes:
    proc_mask = table.index.get_level_values("Process") == proc
    proc_rows = table[proc_mask]
    
    for col in table.columns:
        vals = proc_rows[col].dropna()
        if vals.empty:
            for idx in proc_rows.index:
                str_table.loc[idx, col] = ""
            continue
        
        best_val = vals.max() if _is_higher_better(col) else vals.min()
        
        for idx in proc_rows.index:
            val = table.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ""
                continue
            
            fmt = f"{val:.3f}"
            str_table.loc[idx, col] = (
                r"\textbf{" + fmt + r"}"
                if abs(val - best_val) < 1e-6
                else fmt
            )

# ── Column format ─────────────────────────────────────────────────────────────
n_metrics = len(table.columns)
col_format = "ll|" + "|".join(["c"] * n_metrics)

# ── Generate LaTeX ────────────────────────────────────────────────────────────
latex = str_table.to_latex(
    multicolumn=True,
    multicolumn_format="c",
    multirow=True,
    escape=False,
    column_format=col_format,
    caption=(
        r"Median test results grouped by Process and Approach. "
        r"Metrics shown: MAE\_TEST, RMSE\_TEST, and WAPE\_TEST. "
        r"Lower values are better. "
        r"\textbf{Bold} indicates the best value per process and metric."
    ),
    label="tab:energy_results_median",
    position="H",
)

# Remove \cline lines
latex = re.sub(r"\s*\\cline\{[^}]+\}", "", latex)

# Insert \midrule between process groups
latex_lines = latex.split("\n")
new_lines = []
proc_count = 0

for line in latex_lines:
    if re.search(r"\\multirow", line):
        if proc_count > 0:
            new_lines.append(r"\midrule")
        proc_count += 1
    new_lines.append(line)

latex = "\n".join(new_lines)

print(latex.replace("_", r"\_"))

\begin{table}[H]
\caption{Median test results grouped by Process and Approach. Metrics shown: MAE\\_TEST, RMSE\\_TEST, and WAPE\\_TEST. Lower values are better. \textbf{Bold} indicates the best value per process and metric.}
\label{tab:energy\_results\_median}
\begin{tabular}{ll|c|c|c}
\toprule
 &  & MAE\_TEST & RMSE\_TEST & WAPE\_TEST \\
Process & Approach &  &  &  \\
\midrule
process\_1 & Baseline & \textbf{0.393} & \textbf{0.398} & \textbf{55.836} \\
\multirow[t]{7}{*}{process\_2} & Baseline & 98.483 & 113.056 & 153.153 \\
 & DTW + Ext. Factors + Prev Act & 113.089 & 122.676 & 49.514 \\
 & DTW + Ext. Factors + Prev Act (autoreg) & 113.850 & 123.375 & \textbf{46.219} \\
 & DTW + Seq2Seq & \textbf{79.954} & 98.107 & 60.940 \\
 & DTW + Seq2Seq + Ext. Factors + Prev Act & 81.727 & 100.374 & 76.498 \\
 & DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg) & 81.748 & 100.381 & 73.867 \\
 & DTW + pos & 87.846 & \textbf{97.266} & 52.325 \\
\midrule
\multirow[t]{7}{*}{process\_3} & Baseline & 

In [28]:
df_energy_results = pd.read_parquet(run_dir / 'summary_train_test.parquet')

#df_energy_results = df_energy_results[['Approach', 'MAE_TEST', 'RMSE_TEST', 'WAPE_TEST']]

df_energy_results

,Process,Sensor,Approach,MAE_TRAIN,RMSE_TRAIN,WAPE_TRAIN,R2_TRAIN,MAE_TEST,RMSE_TEST,WAPE_TEST,R2_TEST
0,process_1,autoclave_cooling_water_demand_kW_energy_to_model,Baseline,NaN,NaN,NaN,NaN,10.8275,13.8469,34.3086,-0.1674
1,process_1,autoclave_steam_demand_kW_energy_to_model,Baseline,NaN,NaN,NaN,NaN,33.4937,38.5449,55.8359,-0.3345
2,process_1,bottling_power_kW_energy_to_model,Baseline,NaN,NaN,NaN,NaN,0.0000,0.0000,NaN,1.0000
3,process_1,destillation_steam_demand_kW_energy_to_model,Baseline,NaN,NaN,NaN,NaN,0.0000,0.0000,NaN,1.0000
4,process_1,packaging_power_kW_energy_to_model,Baseline,NaN,NaN,NaN,NaN,0.7854,0.7951,298.2858,-3.5218
...,...,...,...,...,...,...,...,...,...,...,...
393,process_4,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Ext. Factors + Prev Act (autoreg),0.6424,0.7303,1.0915,-1.1511,0.8194,0.9666,1.3622,-1.7357
394,process_4,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Seq2Seq,0.7759,0.8836,1.2883,-2.2029,0.7716,0.9282,1.3142,-2.4513
395,process_4,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Seq2Seq + Ext. Factors + Prev Act,1.0790,1.2131,1.8163,-3.6276,1.0421,1.1873,1.8016,-2.6697
396,process_4,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg),1.0792,1.2138,1.8166,-3.6266,1.0427,1.1863,1.8026,-2.6696


In [29]:
df_energy_results['Approach'].unique()

array(['Baseline', 'DTW + Ext. Factors + Prev Act',
       'DTW + Ext. Factors + Prev Act (autoreg)', 'DTW + Seq2Seq',
       'DTW + Seq2Seq + Ext. Factors + Prev Act',
       'DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg)', 'DTW + pos'],
      dtype=object)

In [30]:
df_energy_results['Process'].value_counts()

Process
process_4    196
process_3    161
process_2     35
process_1      6
Name: count, dtype: int64